In [ ]:
# Testing new code
import pickle
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

def load_dataset(filepath):
    """Load a dataset from a pickle file and return as a DataFrame."""
    with open(filepath, "rb") as f:
        results = pickle.load(f)
    return pd.DataFrame(results)

def extract_visit_data(df, start_time_str):
    """Extract bee visits per minute per x-coordinate during the 1-minute window starting at start_time_str."""
    sim_start_time = datetime.strptime("09:00:00", "%H:%M:%S")
    t_start = int((datetime.strptime(start_time_str, "%H:%M:%S") - sim_start_time).total_seconds())
    t_end = t_start + 60  # fixed 1-minute window
    window_minutes = 1.0  # always 1 minute

    # Filter for plant agents within the timestep interval
    plant_df = df[
        (df["Agent Type"] == "plant") &
        (df["Step"] >= t_start) &
        (df["Step"] < t_end)
    ].copy()

    # Convert positions if necessary
    if isinstance(plant_df["Bee Positions"].iloc[0], str):
        plant_df["Bee Positions"] = plant_df["Bee Positions"].apply(eval)
    plant_df["x"] = plant_df["Bee Positions"].apply(lambda pos: pos[0])

    # Ensure visit_count is numeric
    plant_df["Plant visit count"] = pd.to_numeric(plant_df["Plant visit count"], errors="coerce")

    # Create plant_id (if no explicit ID, use Bee Positions tuple)
    plant_df["plant_id"] = plant_df["Bee Positions"].apply(lambda pos: tuple(pos))

    # Group by RunId, x, plant_id → get max and min visit count in time window
    visit_stats = (
        plant_df.groupby(["RunId", "x", "plant_id"])["Plant visit count"]
        .agg(["max", "min"])
        .reset_index()
    )

    # Compute number of new visits per plant in window
    visit_stats["new_visits"] = visit_stats["max"] - visit_stats["min"]

    # Sum new visits across plants at each x, for each RunId
    visits_per_x_run = (
        visit_stats.groupby(["RunId", "x"])["new_visits"]
        .sum()
        .reset_index()
    )

    # Convert to visits per minute (here window_minutes = 1.0, so this is just new_visits)
    visits_per_x_run["visits_per_minute"] = visits_per_x_run["new_visits"] / window_minutes

    return visits_per_x_run



def plot_comparison(interval_data_dict, start_time_str):
    """Plot comparison of bee visits per minute across multiple datasets with 95% CI."""
    plt.figure(figsize=(12, 6))
    sns.set(style="whitegrid", font_scale=1.2)  # Optional nice styling

    # Infer number of simulations from one dataset
    num_simulations = len(next(iter(interval_data_dict.values()))["RunId"].unique())

    start_time_dt = datetime.strptime(start_time_str, "%H:%M:%S")
    start_time_formatted = start_time_dt.strftime("%H:%M")  # format as "HH:MM"
    
    for label, visit_counts in interval_data_dict.items():
        sns.lineplot(data=visit_counts, x="x", y="visits_per_minute", errorbar=("ci", 95), label=label)
        # sns.scatterplot(data=visit_counts, x="x", y="visits_per_minute", label=label)
    plt.title(
        f"Average number of bee visits per minute among the polytunnel at {start_time_formatted} "
        f"across {num_simulations} simulations (95% CI)"
    )
    plt.xlabel("Plant X Coordinate")
    plt.ylabel("Average Number of Bee Visits per Minute (with 95% CI)")
    plt.grid(True)
    plt.tight_layout()
    plt.show()


# Loading and processing
First load the dataset, then declare which period of the dataset should be visualized

In [ ]:
df1 = load_dataset("batch_results_pure_levy.pkl")
print("Loaded dataset pure levy")
pure_levy_data1 = extract_visit_data(df1, "09:00:00")
pure_levy_data2 = extract_visit_data(df1, "11:30:00")
pure_levy_data3 = extract_visit_data(df1, "13:59:00")



In [ ]:
df2 = load_dataset("batch_results_truncated_levy.pkl")
print("Loaded dataset truncated levy")
truncated_levy_data1 = extract_visit_data(df2, "09:00:00")
truncated_levy_data2 = extract_visit_data(df2, "11:30:00")
truncated_levy_data3 = extract_visit_data(df2, "13:59:00")


In [ ]:
df3 = load_dataset("batch_results_composite_levy.pkl")
print("Loaded dataset composite levy")
composite_levy_data1 = extract_visit_data(df3, "09:00:00")
composite_levy_data2 = extract_visit_data(df3, "11:30:00")
composite_levy_data3 = extract_visit_data(df3, "13:59:00")


# Plot the graphs at different times

In [ ]:
plot_comparison({
    "Pure Lévy": pure_levy_data1,
    "Truncated Lévy": truncated_levy_data1,
    "Composite Lévy": composite_levy_data1
}, "09:00:00")

In [ ]:
plot_comparison({
    "Pure Lévy": pure_levy_data2,
    "Truncated Lévy": truncated_levy_data2,
    "Composite Lévy": composite_levy_data2
}, "11:30:00")

In [ ]:
plot_comparison({
    "Pure Lévy": pure_levy_data3,
    "Truncated Lévy": truncated_levy_data3,
    "Composite Lévy": composite_levy_data3
}, "13:59:00")

# Animating Bee Visitation Rate Per Minute

In [ ]:
%matplotlib notebook

import pickle
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import seaborn as sns
from datetime import datetime, timedelta
from IPython.display import HTML

def load_dataset(filepath):
    with open(filepath, "rb") as f:
        results = pickle.load(f)
    return pd.DataFrame(results)

def extract_visit_data(df, start_time_str):
    sim_start_time = datetime.strptime("09:00:00", "%H:%M:%S")
    t_start = int((datetime.strptime(start_time_str, "%H:%M:%S") - sim_start_time).total_seconds())
    t_end = t_start + 60
    window_minutes = 1.0

    plant_df = df[
        (df["Agent Type"] == "plant") & (df["Step"] >= t_start) & (df["Step"] < t_end)
    ].copy()

    if isinstance(plant_df["Bee Positions"].iloc[0], str):
        plant_df["Bee Positions"] = plant_df["Bee Positions"].apply(eval)
    plant_df["x"] = plant_df["Bee Positions"].apply(lambda pos: pos[0])
    plant_df["Plant visit count"] = pd.to_numeric(plant_df["Plant visit count"], errors="coerce")
    plant_df["plant_id"] = plant_df["Bee Positions"].apply(lambda pos: tuple(pos))

    visit_stats = (
        plant_df.groupby(["RunId", "x", "plant_id"])["Plant visit count"]
        .agg(["max", "min"])
        .reset_index()
    )

    visit_stats["new_visits"] = visit_stats["max"] - visit_stats["min"]

    visits_per_x_run = (
        visit_stats.groupby(["RunId", "x"])["new_visits"]
        .sum()
        .reset_index()
    )

    visits_per_x_run["visits_per_minute"] = visits_per_x_run["new_visits"] / window_minutes
    return visits_per_x_run

def precompute_visit_counts(interval_data_dict, df_sample, interval=600):
    precomputed_visit_counts = {}
    all_visits = []

    start_time_seconds = 0
    end_time_seconds = df_sample["Step"].max() + 1

    for label, df in interval_data_dict.items():
        precomputed_visit_counts[label] = {}
        for current_time_seconds in range(start_time_seconds, end_time_seconds, interval):
            current_time = datetime.strptime("09:00:00", "%H:%M:%S") + timedelta(seconds=current_time_seconds)
            current_time_str = current_time.strftime("%H:%M:%S")
            visit_counts = extract_visit_data(df, current_time_str)
            precomputed_visit_counts[label][current_time_seconds] = visit_counts
            all_visits.extend(visit_counts["visits_per_minute"].values)

    global_ymin = 0
    global_ymax = max(all_visits) * 1.1 if all_visits else 1

    return precomputed_visit_counts, start_time_seconds, end_time_seconds, global_ymin, global_ymax

def animate_visits_precomputed(precomputed_visit_counts, start_time_seconds, end_time_seconds, global_ymin, global_ymax, interval=600):
    fig, ax = plt.subplots(figsize=(12, 6))
    sns.set_theme(style="whitegrid")
    keys_list = list(precomputed_visit_counts.keys())
    num_simulations = len(next(iter(precomputed_visit_counts[keys_list[0]].values()))["RunId"].unique())

    def update(frame_idx):
        ax.clear()
        current_time_seconds = start_time_seconds + frame_idx * interval
        current_time = datetime.strptime("09:00:00", "%H:%M:%S") + timedelta(seconds=current_time_seconds)
        time_str_for_title = current_time.strftime("%H:%M")

        for label in precomputed_visit_counts.keys():
            visit_counts = precomputed_visit_counts[label][current_time_seconds]
            sns.lineplot(data=visit_counts, x="x", y="visits_per_minute", errorbar=("ci", 95), label=label, ax=ax)

        ax.set_ylim(global_ymin, global_ymax)
        ax.set_title(
            f"Mean number of bee visits in one-minute window starting at {time_str_for_title} across {num_simulations} simulations (95% CI)"
        )
        ax.set_xlabel("Polytunnel length (m)")
        ax.set_ylabel("Number of Bee Visits in One Minute Window")
        ax.grid(True)
        ax.legend()

    total_frames = (end_time_seconds - start_time_seconds) // interval
    anim = animation.FuncAnimation(fig, update, frames=total_frames, interval=500, repeat=False)
    return anim, HTML(anim.to_jshtml())


In [ ]:
# Load your datasets
df_pure = load_dataset("batch_results_pure_levy.pkl")
df_truncated = load_dataset("batch_results_truncated_levy.pkl")
df_composite = load_dataset("batch_results_composite_levy.pkl")
df_row = load_dataset("batch_results_row_following.pkl")
interval_data_dict = {
    "pure_levy": df_pure,
    "truncated_levy": df_truncated,
    "composite_levy": df_composite,
    "biased_composite_levy": df_row  # Uncomment if you want to include row following data
}

# Precompute 10-minute averages
precomputed_visit_counts, start_sec, end_sec, global_ymin, global_ymax = precompute_visit_counts(interval_data_dict, df_pure, interval=600)

del df_pure, df_truncated, df_composite  # Free memory

# Animate!
anim, html_obj = animate_visits_precomputed(
    precomputed_visit_counts, start_sec, end_sec, global_ymin, global_ymax, interval=600
)

In [ ]:
html_obj

In [ ]:
# Saving the animation to HTML file
with open("bee_visitation_animation.html", "w") as f:
    f.write(anim.to_jshtml())

In [ ]:
# For visualization in a web browser
import webbrowser
webbrowser.open("bee_visitation_animation.html")